# Sub-Saharan African Bat Occurrence Records with Taxonomy, Locality, Coordinates, and Specimen Metadata Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In this example, we use the FAIR^2 Croissant description for a bat occurrence dataset from sub-Saharan Africa. The notebook follows a step-by-step workflow:

1. Load the Croissant metadata.
2. Inspect the dataset structure, including record sets and fields by `@id`.
3. Attempt to load records from each record set dynamically.
4. Perform lightweight exploratory data analysis when tabular records are available.
5. Create visual summaries for any loaded tables.

> Important: all dataset entities should be referenced by their Croissant `@id` when available.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import json
import math
import warnings

import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid")

# Define the dataset URL
url = "https://sen.science/doi/10.71728/senscience.0tcs-4srv/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_obj = dataset.metadata

# Convert metadata to a JSON-like object for convenient inspection
metadata_json = metadata_obj.to_json()

dataset_name = getattr(metadata_obj, "name", metadata_json.get("name", "Unknown dataset"))
dataset_description = getattr(metadata_obj, "description", metadata_json.get("description", "No description available."))

print(f"Dataset: {dataset_name}")
print(f"Description: {dataset_description}")
print(f"Croissant URL: {url}")

In [ ]:
# Display a compact metadata summary without subscripting the metadata object itself
summary_fields = [
    "name",
    "description",
    "identifier",
    "license",
    "version",
    "datePublished",
    "temporalCoverage",
    "spatialCoverage",
]

summary = []
for field_name in summary_fields:
    value = getattr(metadata_obj, field_name, metadata_json.get(field_name))
    summary.append({"field": field_name, "value": value})

pd.DataFrame(summary)

## 2. Data Overview
Review available record sets, fields, and their IDs.

The FAIR^2 package metadata included with this prompt indicates that the top-level dataset currently exposes an empty `recordSet` list. This means there may be no directly declared Croissant record sets available from the schema at this URL.

Even so, we still inspect the metadata programmatically and build a reusable workflow that works whether record sets are present now or added in a future version.

In [ ]:
# Inspect the top-level metadata JSON keys
top_level_keys = sorted(list(metadata_json.keys()))
print("Top-level metadata keys:")
print(top_level_keys)

In [ ]:
# Extract record set information from the metadata JSON
record_sets_json = metadata_json.get("recordSet", []) or []

record_set_rows = []
for rs in record_sets_json:
    rs_id = rs.get("@id", "<missing_recordset_id>")
    rs_name = rs.get("name", "")
    rs_desc = rs.get("description", "")
    field_count = len(rs.get("field", []) or [])
    record_set_rows.append({
        "record_set_@id": rs_id,
        "name": rs_name,
        "description": rs_desc,
        "field_count": field_count,
    })

record_sets_df = pd.DataFrame(record_set_rows)

if record_sets_df.empty:
    print("No record sets were declared in the Croissant metadata at this URL.")
else:
    display(record_sets_df)

In [ ]:
# Build a field-level overview using record set and field @id values
field_rows = []

for rs in record_sets_json:
    rs_id = rs.get("@id", "<missing_recordset_id>")
    for field in rs.get("field", []) or []:
        field_rows.append({
            "record_set_@id": rs_id,
            "field_@id": field.get("@id", "<missing_field_id>"),
            "field_name": field.get("name", ""),
            "description": field.get("description", ""),
            "data_type": field.get("dataType", ""),
            "source": json.dumps(field.get("source", {}), ensure_ascii=False) if field.get("source") else "",
        })

fields_df = pd.DataFrame(field_rows)

if fields_df.empty:
    print("No field definitions were found because there are no declared record sets.")
else:
    display(fields_df.head(20))

### Optional: inspect a few records from a specific record set
The template includes iterating over `dataset.records(record_set=<id_of_the_records_set>)`.

Because this dataset currently declares no record sets, the code below first checks whether any `@id` values are available before attempting to read records.

In [ ]:
record_set_ids = [rs.get("@id") for rs in record_sets_json if rs.get("@id")]
print("Record set @ids found:", record_set_ids)

if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"Showing up to 5 records from record set: {example_record_set_id}")
    for i, x in enumerate(dataset.records(record_set=example_record_set_id)):
        print(x)
        if i >= 4:
            break
else:
    print("No record set @id is available, so no records can be previewed from this Croissant description.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

This section is written dynamically so it can run whether the Croissant schema has zero, one, or multiple record sets.

In [ ]:
# Extract data from each record set
record_sets = record_set_ids
dataframes = {}
load_report = []

for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        load_report.append({
            "record_set_@id": record_set,
            "status": "loaded",
            "rows": len(df),
            "columns": len(df.columns),
        })
    except Exception as e:
        load_report.append({
            "record_set_@id": record_set,
            "status": f"error: {type(e).__name__}",
            "rows": None,
            "columns": None,
        })

load_report_df = pd.DataFrame(load_report)
if load_report_df.empty:
    print("No record sets were available to extract.")
else:
    display(load_report_df)

In [ ]:
# Show columns and a sample for the first successfully loaded record set, if any
loaded_record_set_ids = [k for k, v in dataframes.items() if isinstance(v, pd.DataFrame)]

if loaded_record_set_ids:
    selected_record_set_id = loaded_record_set_ids[0]
    print("Selected record set @id:", selected_record_set_id)
    print("Columns:")
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())
else:
    selected_record_set_id = None
    print("No DataFrame could be created because no record set records were available.")

### Fallback metadata tables
If no Croissant record sets are defined, we can still analyze useful metadata-derived tables such as dataset keywords, bias statements, limitations, and use cases.

In [ ]:
keywords_df = pd.DataFrame({"keyword": metadata_json.get("keywords", []) or []})
biases_df = pd.DataFrame({"dataBiases": metadata_json.get("dataBiases", []) or []})
limitations_df = pd.DataFrame({"dataLimitations": metadata_json.get("dataLimitations", []) or []})
use_cases_df = pd.DataFrame({"dataUseCases": metadata_json.get("dataUseCases", []) or []})

print("Keywords table:")
display(keywords_df.head(10))

print("Data biases table:")
display(biases_df)

print("Data limitations table:")
display(limitations_df)

print("Data use cases table:")
display(use_cases_df)

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Because this Croissant schema currently exposes no record sets, the notebook includes two paths:

- **Primary path**: perform EDA on the first loaded record set, if one exists.
- **Fallback path**: perform EDA on metadata-derived tables.

In [ ]:
# Primary EDA path for record-set data, if available
if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id].copy()
    
    numeric_columns = df.select_dtypes(include="number").columns.tolist()
    object_columns = df.select_dtypes(include="object").columns.tolist()
    
    print("Numeric columns:", numeric_columns)
    print("Object/string columns:", object_columns[:20])
    
    if numeric_columns:
        numeric_field = numeric_columns[0]
        threshold = df[numeric_field].dropna().median()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())
        
        std_value = filtered_df[numeric_field].std()
        if pd.notna(std_value) and std_value != 0:
            filtered_df[f"{numeric_field}_normalized"] = (
                filtered_df[numeric_field] - filtered_df[numeric_field].mean()
            ) / std_value
        else:
            filtered_df[f"{numeric_field}_normalized"] = 0
        
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        group_field = object_columns[0] if object_columns else None
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head(10).reset_index(name=f"mean_{numeric_field}"))
    else:
        print("No numeric columns are available in the selected record set for numeric EDA.")
else:
    print("Skipping primary record-level EDA because no record-set table was loaded.")

In [ ]:
# Fallback EDA on metadata-derived tables
# Example: analyze text length statistics for descriptive metadata lists

metadata_text_tables = {
    "keyword": keywords_df.rename(columns={"keyword": "text"}).copy(),
    "dataBiases": biases_df.rename(columns={"dataBiases": "text"}).copy(),
    "dataLimitations": limitations_df.rename(columns={"dataLimitations": "text"}).copy(),
    "dataUseCases": use_cases_df.rename(columns={"dataUseCases": "text"}).copy(),
}

eda_rows = []
for table_name, table_df in metadata_text_tables.items():
    if not table_df.empty:
        table_df["text_length"] = table_df["text"].astype(str).str.len()
        threshold = table_df["text_length"].median()
        filtered_df = table_df[table_df["text_length"] > threshold].copy()
        std_value = filtered_df["text_length"].std()
        if pd.notna(std_value) and std_value != 0:
            filtered_df["text_length_normalized"] = (
                filtered_df["text_length"] - filtered_df["text_length"].mean()
            ) / std_value
        else:
            filtered_df["text_length_normalized"] = 0
        
        eda_rows.append({
            "table": table_name,
            "count": len(table_df),
            "median_text_length": threshold,
            "mean_text_length": table_df["text_length"].mean(),
            "max_text_length": table_df["text_length"].max(),
        })

eda_summary_df = pd.DataFrame(eda_rows)
display(eda_summary_df)

if not biases_df.empty:
    biases_eda = biases_df.rename(columns={"dataBiases": "text"}).copy()
    biases_eda["text_length"] = biases_eda["text"].astype(str).str.len()
    threshold = biases_eda["text_length"].median()
    filtered_df = biases_eda[biases_eda["text_length"] > threshold].copy()
    std_value = filtered_df["text_length"].std()
    if pd.notna(std_value) and std_value != 0:
        filtered_df["text_length_normalized"] = (
            filtered_df["text_length"] - filtered_df["text_length"].mean()
        ) / std_value
    else:
        filtered_df["text_length_normalized"] = 0
    
    print(f"Filtered records with text_length > {threshold}:")
    display(filtered_df)
    
    group_field = "length_category"
    filtered_df[group_field] = pd.cut(
        filtered_df["text_length"],
        bins=3,
        labels=["shorter", "medium", "longer"],
        include_lowest=True
    )
    grouped_df = filtered_df.groupby(group_field)["text_length"].mean().reset_index(name="mean_text_length")
    print(f"Grouped data by {group_field}:")
    display(grouped_df)

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization path 1: plot record-set numeric distributions if available
if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id].copy()
    numeric_columns = df.select_dtypes(include="number").columns.tolist()
    
    if numeric_columns:
        numeric_field = numeric_columns[0]
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field} in {selected_record_set_id}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()
    else:
        print("No numeric columns available for record-set visualization.")
else:
    print("No record-set table available, using metadata-based visualizations instead.")

In [ ]:
# Visualization path 2: metadata-based visualizations

if not keywords_df.empty:
    keywords_plot_df = keywords_df.copy()
    keywords_plot_df["length"] = keywords_plot_df["keyword"].astype(str).str.len()
    keywords_plot_df = keywords_plot_df.sort_values("length", ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=keywords_plot_df, y="keyword", x="length", palette="viridis")
    plt.title("Keyword length overview")
    plt.xlabel("Character count")
    plt.ylabel("Keyword")
    plt.tight_layout()
    plt.show()

text_length_frames = []
for label, df_ in {
    "Biases": biases_df.rename(columns={"dataBiases": "text"}),
    "Limitations": limitations_df.rename(columns={"dataLimitations": "text"}),
    "Use cases": use_cases_df.rename(columns={"dataUseCases": "text"}),
}.items():
    if not df_.empty:
        temp = df_.copy()
        temp["category"] = label
        temp["text_length"] = temp["text"].astype(str).str.len()
        text_length_frames.append(temp[["category", "text_length"]])

if text_length_frames:
    combined_text_lengths = pd.concat(text_length_frames, ignore_index=True)
    
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=combined_text_lengths, x="category", y="text_length", palette="Set2")
    plt.title("Text length distribution across metadata sections")
    plt.xlabel("Section")
    plt.ylabel("Text length")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to work with a Croissant dataset using `mlcroissant`, while keeping references aligned with Croissant `@id` values whenever record sets and fields are available.

### Key observations
- The dataset metadata loaded successfully from the provided Croissant URL.
- The dataset describes a curated bat occurrence resource for sub-Saharan Africa with rich narrative metadata.
- At the time of this notebook, the Croissant metadata appears to declare **no record sets** (`recordSet` is empty), so direct record extraction is not available from the schema.
- Even without record sets, the metadata itself supports useful inspection of dataset scope, stated biases, limitations, and intended use cases.

### Recommended next steps
- Re-check the Croissant schema in case a future version adds `recordSet` entries.
- If the underlying distribution file becomes accessible through a declared record set, reuse the dynamic extraction cells in this notebook.
- Use the metadata-derived warnings on bias, missingness, taxonomic uncertainty, and spatial precision when designing analyses.

Overall, this notebook provides a robust starting point for both current metadata exploration and future record-level analysis.